# Baseline Comparison — Does the VAE Earn Its Complexity?

Input: `data/processed/windows_cc1/X_*.npy` (same whitened PCA features the VAE
uses — a fair comparison isolates the *model*, not the *features*).

**Why this notebook exists:** every result so far validates the VAE+PCA approach
against itself (ablations, thresholding variants) — never against something
simpler. This is the most likely first question an examiner asks: *"how do you
know this needed a VAE?"*

**Scope: CC1 (in-distribution) + CC2 (drift) only** — matches the project's
established evaluation scope (`vae_eval.ipynb`: "SC1/SC2 dropped from scope per
project decision").

**Two baselines, same leak-free protocol as `vae_eval.ipynb`** (threshold chosen
only from `cc1_val`'s unsupervised false-positive rate, never touching test/drift
labels):

1. **Standard (non-variational) Autoencoder** — same encoder/decoder capacity as
   the VAE (`input_dim -> hidden1 -> hidden2 -> latent_dim -> hidden2 -> hidden1
   -> input_dim`, identical `hidden1`/`hidden2`/`latent_dim` pulled from
   `vae_cc1_meta.pkl`), trained on the same `cc1_train` data with the same
   optimizer/epochs/batch size — but with a deterministic bottleneck (no
   `fc_mu`/`fc_lv` split, no reparameterization, no KL term). This isolates
   exactly what the VAE's probabilistic regularization buys over a plain AE of
   identical size and training budget.
2. **Isolation Forest** — a standard, no-deep-learning classical anomaly
   detector (`sklearn.ensemble.IsolationForest`), fit on `cc1_train` only, same
   whitened PCA features.

In [1]:
import numpy as np
import pickle, os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix,
)

torch.manual_seed(42)
np.random.seed(42)

BASE     = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
WIN_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

SETS = ['cc1_train', 'cc1_val', 'cc1_test', 'drift_cc2']
DRIFT_SETS = ['drift_cc2']

X, y = {}, {}
for name in SETS:
    X[name] = np.load(os.path.join(WIN_DIR, f'X_{name}.npy'))
    y[name] = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    print(f'  {name:10s}: {X[name].shape}  anomalies={int(y[name].sum()):,}')

vae_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
vae_meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
print('\nVAE reference results + metadata loaded for comparison.')

  cc1_train : (154198, 26)  anomalies=0
  cc1_val   : (21573, 26)  anomalies=0
  cc1_test  : (44185, 26)  anomalies=256
  drift_cc2 : (76977, 26)  anomalies=720

VAE reference results + metadata loaded for comparison.


## Baseline A — Standard (Non-Variational) Autoencoder

Same `input_dim -> hidden1 -> hidden2 -> latent_dim -> hidden2 -> hidden1 ->
input_dim` capacity as the VAE (pulled from `vae_cc1_meta.pkl`, not
re-tuned), trained on `cc1_train` only with the same optimizer/batch
size/max-epochs/early-stopping-patience as `train_vae.ipynb`. The only
architectural difference: a deterministic bottleneck — no `fc_mu`/`fc_lv`
split, no reparameterization noise, no KL term, loss is pure reconstruction
MSE. This isolates what the VAE's probabilistic latent space specifically
buys over a plain AE of identical size and training budget.

In [2]:
class AE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2),   nn.ReLU(),
            nn.Linear(hidden2, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1),    nn.ReLU(),
            nn.Linear(hidden1, input_dim),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.decoder(self.encoder(x))

    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        return ((self.forward(x) - x) ** 2).mean(dim=1)


def train_ae(input_dim, hidden1, hidden2, latent_dim, X_train_t, X_val_t,
             max_epochs=300, patience=20, lr=1e-3, batch_size=512, seed=42):
    torch.manual_seed(seed)
    model = AE(input_dim, hidden1, hidden2, latent_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=batch_size, shuffle=True)

    history = {'train_loss': [], 'val_loss': []}
    best_val, best_state, patience_ctr = float('inf'), None, 0

    for epoch in range(max_epochs):
        model.train()
        tr_loss, n = 0.0, 0
        for (xb,) in loader:
            opt.zero_grad()
            recon = model(xb)
            loss = nn.functional.mse_loss(recon, xb, reduction='mean')
            loss.backward()
            opt.step()
            bs = xb.size(0)
            tr_loss += loss.item() * bs; n += bs
        tr_loss /= n

        model.eval()
        with torch.no_grad():
            vloss = nn.functional.mse_loss(model(X_val_t), X_val_t, reduction='mean').item()

        history['train_loss'].append(tr_loss); history['val_loss'].append(vloss)

        if vloss < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'    early stop at epoch {epoch + 1} (best val_loss={best_val:.5f})')
                break

    model.load_state_dict(best_state)
    return model, history, best_val


X_train_t = torch.from_numpy(np.clip(X['cc1_train'], -vae_meta['clip'], vae_meta['clip']).astype(np.float32))
X_val_t   = torch.from_numpy(np.clip(X['cc1_val'],   -vae_meta['clip'], vae_meta['clip']).astype(np.float32))

print(f"Training AE: input_dim={vae_meta['input_dim']} hidden1={vae_meta['hidden1']} "
      f"hidden2={vae_meta['hidden2']} latent_dim={vae_meta['latent_dim']} (matching VAE capacity) ...")
ae_model, ae_history, ae_best_val = train_ae(
    vae_meta['input_dim'], vae_meta['hidden1'], vae_meta['hidden2'], vae_meta['latent_dim'],
    X_train_t, X_val_t, max_epochs=300, patience=20, lr=1e-3, batch_size=512,
)
print(f'AE training done. Best val loss: {ae_best_val:.5f}  (epochs run: {len(ae_history["train_loss"])})')

CLIP = vae_meta['clip']
Xc = {name: np.clip(X[name], -CLIP, CLIP).astype(np.float32) for name in SETS}
scores_ae = {}
for name in SETS:
    with torch.no_grad():
        scores_ae[name] = ae_model.anomaly_score(torch.from_numpy(Xc[name])).numpy()

ae_model_path = os.path.join(MODEL_DIR, 'ae_cc1.pt')
torch.save(ae_model.state_dict(), ae_model_path)
ae_meta = {
    'input_dim': vae_meta['input_dim'], 'hidden1': vae_meta['hidden1'],
    'hidden2': vae_meta['hidden2'], 'latent_dim': vae_meta['latent_dim'],
    'clip': CLIP, 'best_val_loss': ae_best_val,
    'mu_train': float(scores_ae['cc1_train'].mean()), 'sigma_train': float(scores_ae['cc1_train'].std()),
}
with open(os.path.join(MODEL_DIR, 'ae_cc1_meta.pkl'), 'wb') as f:
    pickle.dump(ae_meta, f)
print(f'Saved AE checkpoint -> {ae_model_path}')
print(f'Saved AE metadata   -> {os.path.join(MODEL_DIR, "ae_cc1_meta.pkl")}')

Training AE: input_dim=26 hidden1=64 hidden2=32 latent_dim=32 (matching VAE capacity) ...


    early stop at epoch 139 (best val_loss=0.00017)
AE training done. Best val loss: 0.00017  (epochs run: 139)


Saved AE checkpoint -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models\ae_cc1.pt
Saved AE metadata   -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models\ae_cc1_meta.pkl


## Baseline B — Isolation Forest

Fit on `cc1_train` only (same training set the VAE uses), same whitened PCA
features. `n_estimators=100` (sklearn default-adjacent), `contamination='auto'`
(doesn't use labels — purely structural).

In [3]:
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(Xc['cc1_train'])

# decision_function: higher = more normal. Negate so higher = more anomalous, matching every other score in this project.
scores_iso = {name: -iso_forest.decision_function(Xc[name]) for name in SETS}
print('Isolation Forest fit on cc1_train.')

Isolation Forest fit on cc1_train.


## Leak-free thresholds for each baseline (from `cc1_val` only, same discipline as `vae_eval.ipynb`)

In [4]:
thresh_ae  = float(np.percentile(scores_ae['cc1_val'], 99))
thresh_iso = float(np.percentile(scores_iso['cc1_val'], 99))
print(f'Autoencoder val_p99 threshold:       {thresh_ae:.5f}')
print(f'Isolation Forest val_p99 threshold:  {thresh_iso:.4f}')

Autoencoder val_p99 threshold:       0.00120
Isolation Forest val_p99 threshold:  0.0375


## Full comparison: VAE vs. Autoencoder vs. Isolation Forest, on CC1 (`cc1_test`) and CC2 (`drift_cc2`)

In [5]:
def evaluate(scores, y_true, threshold):
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr  = average_precision_score(y_true, scores)
    pred = (scores > threshold).astype(int)
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'precision': p, 'recall': r, 'f1': f1}

results = {'autoencoder': {}, 'isolation_forest': {}, 'vae': {}}
for name in ['cc1_test'] + DRIFT_SETS:
    results['autoencoder'][name] = evaluate(scores_ae[name], y[name], thresh_ae)
    results['isolation_forest'][name] = evaluate(scores_iso[name], y[name], thresh_iso)
    results['vae'][name] = {
        'auc_roc': vae_eval['auc'][name]['auc_roc'],
        'auc_pr':  vae_eval['auc'][name]['auc_pr'],
        'precision': vae_eval['precision_recall'][name]['val_p99']['precision'],
        'recall':    vae_eval['precision_recall'][name]['val_p99']['recall'],
        'f1':        vae_eval['precision_recall'][name]['val_p99']['f1'],
    }

print(f'{"set":12s} {"method":18s} {"AUC-ROC":>9s} {"AUC-PR":>8s} {"precision":>10s} {"recall":>8s} {"F1":>7s}')
for name in ['cc1_test'] + DRIFT_SETS:
    for method in ['autoencoder', 'isolation_forest', 'vae']:
        r = results[method][name]
        print(f'{name:12s} {method:18s} {r["auc_roc"]:9.4f} {r["auc_pr"]:8.4f} {r["precision"]:10.3f} {r["recall"]:8.3f} {r["f1"]:7.3f}')
    print()

set          method               AUC-ROC   AUC-PR  precision   recall      F1
cc1_test     autoencoder           0.9592   0.4130      0.264    0.535   0.354
cc1_test     isolation_forest      0.7584   0.1139      0.140    0.230   0.174
cc1_test     vae                   0.8763   0.6014      0.630    0.605   0.618

drift_cc2    autoencoder           0.8969   0.0787      0.092    0.575   0.159
drift_cc2    isolation_forest      0.8348   0.2518      0.146    0.444   0.220
drift_cc2    vae                   0.8812   0.4089      0.168    0.772   0.276



## Save results

In [6]:
out_path = os.path.join(MODEL_DIR, 'baseline_comparison.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models\baseline_comparison.pkl


## How to read this

If the VAE's F1/AUC-PR clearly beats both baselines on `cc1_test` (in-distribution),
that justifies the added complexity for the core task. If a baseline is
competitive or better on `drift_cc2` specifically, that's worth reporting honestly
— it would mean the VAE's nonlinear/probabilistic modeling isn't adding value there
specifically, which is itself a legitimate, useful finding (not a failure of this
notebook). The Autoencoder result in particular isolates the VAE's KL-regularized
latent space as the variable of interest: same capacity, same training budget,
same data — the only difference is the probabilistic bottleneck.